# ElasticNet pool model
Global ElasticNet (L1+L2) trained on pooled data. Features: **VIX 5-day SMA**, **Fear & Greed change** (lag1 − lag5), month_sin, month_cos. Multi-output: 21-step returns, scaled inputs.

In [7]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.multioutput import MultiOutputRegressor

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

ELASTICNET_ALPHA = 0.1
ELASTICNET_L1_RATIO = 0.5  # 0 = Ridge, 1 = Lasso

In [8]:
def build_feature_df(grp: pd.DataFrame):
    """Features: VIX 5-day SMA, Fear & Greed change (lag1 - lag5), month sin/cos. Target = next 21 returns."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    if "vix" in df.columns:
        df["vix_sma_5"] = df["vix"].astype(float).shift(1).rolling(5).mean()
    else:
        df["vix_sma_5"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols = ["vix_sma_5", "month_sin", "month_cos", "fear_greed_change"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols, target_cols


def train_global_elasticnet(stacked: pd.DataFrame, horizon: int):
    """Train one ElasticNet (MultiOutputRegressor) on pooled data. Returns dict for predict_elasticnet_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    X = pooled_feat[feature_cols].values.astype(np.float64)
    y = pooled_feat[target_cols].values.astype(np.float64)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    enet = MultiOutputRegressor(ElasticNet(alpha=ELASTICNET_ALPHA, l1_ratio=ELASTICNET_L1_RATIO, random_state=42))
    enet.fit(X_s, y)
    return {"model": enet, "scaler": scaler, "feature_cols": feature_cols}


def predict_elasticnet_global(context_df: pd.DataFrame, horizon: int, global_enet: dict) -> list:
    """Predict 21 price steps using pre-trained global ElasticNet."""
    if global_enet is None:
        return []
    try:
        feat_df, feature_cols, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < 1:
        return []
    X = feat_df[feature_cols].values.astype(np.float64)
    X_s = global_enet["scaler"].transform(X)
    last_row = X_s[-1:]
    pred_returns = global_enet["model"].predict(last_row).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + pred_returns]))[1:]
    return [float(p) for p in prices[:horizon]]

In [9]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-08,AAPL,116.360001,154376600,25.469999,39.080000
1,2021-03-09,AAPL,121.089996,129525800,24.030001,43.360000
2,2021-03-10,AAPL,119.980003,111943300,22.559999,45.560000
3,2021-03-11,AAPL,121.959999,103026500,21.910000,50.480000
4,2021-03-12,AAPL,121.029999,88105100,20.690001,53.720000
5,2021-03-15,AAPL,123.989998,92403800,20.030001,56.520000
6,2021-03-16,AAPL,125.570000,115227900,19.790001,54.800000
7,2021-03-17,AAPL,124.760002,111932600,19.230000,57.866667
8,2021-03-18,AAPL,120.529999,121229700,21.580000,52.333333
9,2021-03-19,AAPL,119.989998,185549500,20.950001,50.833333


In [10]:
global_enet = train_global_elasticnet(stacked, FORECAST_HORIZON)
print("Global ElasticNet trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

Global ElasticNet trained on 11960 pooled train rows.


In [11]:
model_name = "elasticnet"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_elasticnet_global(context_df, FORECAST_HORIZON, global_enet)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-09,277.179993,278.140605,0,AAPL
1,2025-12-10,278.779999,278.395132,0,AAPL
2,2025-12-11,278.029999,278.650857,0,AAPL
3,2025-12-12,278.279999,278.911621,0,AAPL
4,2025-12-15,274.109985,279.168725,0,AAPL


In [12]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_elasticnet_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_elasticnet_pool.parquet")

         model   symbol        MAE       RMSE    MAPE_%
0   elasticnet     AAPL  10.885479  13.197125  4.137882
1   elasticnet     MSFT  26.718009  31.118871  6.262102
2   elasticnet    GOOGL  13.294422  15.097686  4.164246
3   elasticnet     AMZN  13.809339  16.286110  6.324753
4   elasticnet      JPM  10.947172  12.757888  3.524716
5   elasticnet      JNJ   9.414339  10.706682  4.110612
6   elasticnet      WMT   4.864321   5.585700  3.949369
7   elasticnet      SPY   8.385210   9.639440  1.220615
8   elasticnet      XOM   7.540868   8.801154  5.401796
9   elasticnet     NVDA   6.766041   8.094178  3.679970
10  elasticnet  overall  11.262520  15.554825  4.277606
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_elasticnet_pool.parquet
